# PAE - SSL Training Pipeline on Kaggle

Repo: https://github.com/maferozuone/PAE-ssl-project

## Pipeline
1. Configuracion
2. Clonar repo
3. Preparar Food-101
4. Generar subsets SAS
5. Entrenar SSL
6. Guardar resultados

## 0. Configuracion — edita aqui

In [ ]:
# ============================================================
# EDITA ESTOS PARAMETROS ANTES DE EJECUTAR
# ============================================================

GITHUB_REPO = "https://github.com/maferozuone/PAE-ssl-project.git"

# "debug" = rapido para probar | "full" = experimento completo
MODE = "full"

# Metodos SSL: "simsiam", "byol", "cpc", "align_uniform"
SSL_METHODS = ["simsiam", "byol"]

# Subconjuntos: "full", "sas_keep_60pct", "sas_keep_80pct", "random_keep_60pct"
DATA_CONFIGS = ["full", "sas_keep_60pct", "sas_keep_80pct", "random_keep_60pct"]

# Clusters K-Means para SAS no supervisado
N_CLUSTERS = 101

print(f"Modo: {MODE}")
print(f"Metodos SSL: {SSL_METHODS}")
print(f"Data configs: {DATA_CONFIGS}")

## 1. Clonar repositorio

In [ ]:
import os, sys

WORK_DIR = "/kaggle/working/PAE-ssl-project"

if os.path.exists(WORK_DIR):
    print("Repo ya existe, actualizando...")
    os.system(f"git -C {WORK_DIR} pull")
else:
    ret = os.system(f"git clone {GITHUB_REPO} {WORK_DIR}")
    if ret != 0:
        raise RuntimeError("Error al clonar el repo. Verifica que sea publico.")

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)
print(f"Directorio: {os.getcwd()}")
os.system("ls -la")

In [ ]:
os.system("pip install -q timm")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Preparar Food-101

Agrega el dataset desde el panel derecho: **+ Add Data** -> busca **food-101**

In [ ]:
# Detectar ruta de Food-101 en Kaggle
KAGGLE_FOOD101_PATHS = [
    "/kaggle/input/food-101/food-101",
    "/kaggle/input/food101/food-101",
    "/kaggle/input/food-101",
]

food101_src = None
for p in KAGGLE_FOOD101_PATHS:
    if os.path.exists(p):
        food101_src = p
        break

if food101_src is None:
    print("Food-101 NO encontrado. Contenido de /kaggle/input/:")
    os.system("ls /kaggle/input/")
    raise FileNotFoundError("Agrega el dataset Food-101 al notebook (panel derecho -> Add Data)")

print(f"Food-101 encontrado en: {food101_src}")
os.system(f"ls {food101_src}")

In [ ]:
# Crear symlink (evita copiar ~5 GB)
DATA_DIR = os.path.join(WORK_DIR, "data")
FOOD101_LINK = os.path.join(DATA_DIR, "food-101")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(FOOD101_LINK):
    os.symlink(food101_src, FOOD101_LINK)
    print(f"Symlink: {FOOD101_LINK} -> {food101_src}")
else:
    print(f"Symlink ya existe: {FOOD101_LINK}")

os.system(f"ls {FOOD101_LINK}")

In [ ]:
# Parchear config para apuntar a rutas de Kaggle
import config_final
_OriginalConfig = config_final.Config

class KaggleConfig(_OriginalConfig):
    def __init__(self, mode="full"):
        super().__init__(mode)
        self.data_root   = os.path.join(WORK_DIR, "data")
        self.output_root = os.path.join(WORK_DIR, "output")
        self.checkpoint_dir = os.path.join(self.output_root, "checkpoints_final")
        self.subset_dir     = os.path.join(self.output_root, "subsets")
        self.results_dir    = os.path.join(self.output_root, "results_final")
        for d in [self.data_root, self.output_root, self.checkpoint_dir,
                  self.subset_dir, self.results_dir]:
            os.makedirs(d, exist_ok=True)
        if mode == "full":
            self.num_workers = 4
            self.prefetch_factor = 2

config_final.Config = KaggleConfig
config_final.get_config = lambda mode="full": KaggleConfig(mode)

cfg = config_final.get_config(MODE)
print(f"Config: {cfg}")
print(f"  data_root:      {cfg.data_root}")
print(f"  output_root:    {cfg.output_root}")
print(f"  batch_size:     {cfg.batch_size}")
print(f"  epochs_ssl:     {cfg.epochs_ssl}")
print(f"  num_workers:    {cfg.num_workers}")

In [ ]:
from data_utils import load_food101
print("Cargando Food-101...")
dataset = load_food101(cfg, split="train")
print(f"Dataset: {len(dataset)} muestras")

## 3. Generar subconjuntos SAS

In [ ]:
import numpy as np
from data_utils import get_eval_transform
from proxy_model import ProxyModel, compute_embeddings

emb_path    = os.path.join(cfg.subset_dir, f"embeddings_{cfg.mode}.npy")
labels_path = os.path.join(cfg.subset_dir, f"labels_{cfg.mode}.npy")

if os.path.exists(emb_path) and os.path.exists(labels_path):
    print("Cargando embeddings precalculados...")
    embeddings = np.load(emb_path).astype(np.float32)
    labels = np.load(labels_path)
else:
    print("Calculando embeddings proxy (ResNet18 ImageNet)... ~5-10 min")
    proxy = ProxyModel(cfg.proxy_backbone, pretrained=True)
    embeddings, labels = compute_embeddings(
        proxy, dataset,
        transform=get_eval_transform(cfg.image_size),
        device=cfg.device,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers
    )
    embeddings = embeddings.astype(np.float32)
    np.save(emb_path, embeddings)
    np.save(labels_path, labels)
    print(f"Embeddings guardados: {embeddings.shape}")

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
from sas_selection_fast import (
    approximate_latent_classes,
    sas_select_from_labels_fast,
    random_uniform_subset,
)
import time

n_samples = len(dataset)
n_clusters = min(N_CLUSTERS, n_samples)

# Clustering K-Means
latent_labels_path = os.path.join(cfg.subset_dir, f"latent_clusters_k{n_clusters}_{cfg.mode}.npy")
if os.path.exists(latent_labels_path):
    print(f"Cargando clusters K-Means...")
    latent_labels = np.load(latent_labels_path)
else:
    print(f"Agrupando en {n_clusters} clusters K-Means...")
    latent_labels = approximate_latent_classes(embeddings, n_clusters=n_clusters, seed=42)
    np.save(latent_labels_path, latent_labels)
    print(f"Clusters guardados -> {latent_labels_path}")

# Generar subconjuntos
for reduction, keep_frac in zip([10, 20, 40, 60], [0.90, 0.80, 0.60, 0.40]):
    target   = round(n_samples * keep_frac)
    keep_pct = int(keep_frac * 100)
    sas_path  = os.path.join(cfg.subset_dir, f"sas_keep_{keep_pct}pct_{cfg.mode}.npy")
    rand_path = os.path.join(cfg.subset_dir, f"random_keep_{keep_pct}pct_{cfg.mode}.npy")

    if os.path.exists(sas_path):
        print(f"[{keep_pct}%] Ya existe, saltando.")
        continue

    print(f"\n[Reduccion {reduction}%] Seleccionando {target}/{n_samples} muestras...")
    t0 = time.perf_counter()
    result = sas_select_from_labels_fast(
        embeddings, latent_labels, total_budget=target,
        device=cfg.device, refine=False, verbose=True
    )
    random_unif = random_uniform_subset(n_samples, keep_frac, seed=42)
    np.save(sas_path, result["selected_indices"])
    np.save(rand_path, random_unif)
    print(f"  SAS: {len(result['selected_indices'])} muestras [{time.perf_counter()-t0:.1f}s]")

print("\nSubsets disponibles:")
os.system(f"ls -lh {cfg.subset_dir}")

## 4. Entrenamiento SSL

In [ ]:
import train_ssl_final as train_module
import time, json

RESULTS_SUMMARY = []
TOTAL_RUNS = len(SSL_METHODS) * len(DATA_CONFIGS)
run_idx = 0

print(f"Experimentos totales: {TOTAL_RUNS}")
print(f"Metodos: {SSL_METHODS}")
print(f"Data configs: {DATA_CONFIGS}")
print("=" * 60)

for method in SSL_METHODS:
    for data_config in DATA_CONFIGS:
        run_idx += 1
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"{method}_{data_config}_final.pt")
        print(f"\n[{run_idx}/{TOTAL_RUNS}] {method.upper()} / {data_config}")
        print("-" * 50)

        if os.path.exists(ckpt_path):
            print(f"Checkpoint ya existe, saltando.")
            RESULTS_SUMMARY.append({"method": method, "data_config": data_config, "status": "skipped"})
            continue

        t0 = time.perf_counter()
        try:
            train_module.train(method, data_config, mode=MODE, use_dummy=False)
            elapsed = time.perf_counter() - t0
            print(f"Completado en {elapsed/60:.1f} min")
            RESULTS_SUMMARY.append({"method": method, "data_config": data_config,
                                     "status": "done", "elapsed_min": round(elapsed/60, 1)})
        except Exception as e:
            elapsed = time.perf_counter() - t0
            print(f"ERROR: {e}")
            RESULTS_SUMMARY.append({"method": method, "data_config": data_config,
                                     "status": "error", "error": str(e)})

print("\n" + "=" * 60)
print("RESUMEN")
for r in RESULTS_SUMMARY:
    icon = {"done": "OK", "skipped": "--", "error": "!!"}[r['status']]
    mins = f" ({r.get('elapsed_min','?')} min)" if r['status'] == 'done' else ''
    print(f"[{icon}] {r['method']:15s} | {r['data_config']:25s}{mins}")

## 5. Guardar y descargar resultados

In [ ]:
import zipfile, json

# Guardar resumen
summary_path = os.path.join(WORK_DIR, "output", "results_final", "kaggle_run_summary.json")
with open(summary_path, "w") as f:
    json.dump(RESULTS_SUMMARY, f, indent=2)
print(f"Resumen: {summary_path}")

# Empaquetar todo para descarga
output_zip = "/kaggle/working/results_and_checkpoints.zip"
output_dir = os.path.join(WORK_DIR, "output")

print(f"Creando ZIP...")
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, output_dir)
            zf.write(filepath, arcname)

zip_size = os.path.getsize(output_zip) / 1e6
print(f"ZIP listo: {output_zip} ({zip_size:.1f} MB)")
print("Descargalo desde el panel Output de Kaggle.")

In [ ]:
print("Archivos en /kaggle/working/:")
os.system("ls -lh /kaggle/working/")
print("\nCheckpoints:")
os.system(f"ls -lh {cfg.checkpoint_dir} 2>/dev/null || echo '(vacio)'")
print("\nResultados:")
os.system(f"ls -lh {cfg.results_dir} 2>/dev/null || echo '(vacio)'")